In [11]:
from typing import Dict, Any
from occhio.distributions.sparse import SparseUniform
from occhio.autoencoder import TiedLinearRelu
from occhio.toy_model import ToyModel
from occhio.model_grid import ModelGrid, Axis
from torch import Generator, logspace
import torch
from base64 import b64encode

N_FEATURES = 5
N_HIDDEN = 2
EXPERIMENT_SIZE = 100

densities = logspace(0, -2, EXPERIMENT_SIZE)
importances = logspace(-1, 1, EXPERIMENT_SIZE)


g1= Generator(device="cpu").manual_seed(42)
state = g1.get_state()

state_encoded = b64encode(state.numpy().tobytes()).decode("ascii")
print(state_encoded)


KgAAAAAAAAABAAAAAQAAAAAAAAAAAAAAKgAAAAAAAACTijy5AAAAADdEAXEAAAAAUc966AAAAABeraS0AAAAAFGhQvMAAAAAYIfcogAAAACx2yJRAAAAAHjcIewAAAAAkM+2MwAAAADa8yOpAAAAAEPMlcMAAAAATNUn6gAAAAA4bwfcAAAAAFV1ErgAAAAAYtp0aQAAAAAfJK7gAAAAAB07y6IAAAAATeqqrgAAAAA+uPscAAAAAIremlYAAAAA7C//xQAAAABh0N3RAAAAAMGotYsAAAAAB/BQNwAAAADccZOnAAAAALC6LcoAAAAAunPZ2wAAAAAZqcdRAAAAAJWO69wAAAAATIdXZgAAAACAlgtnAAAAAAVqpGQAAAAAtfe1RwAAAAAmDkZiAAAAAIZ0RxoAAAAAAq82UwAAAABUp1AlAAAAAEr4qeUAAAAA9AXtJwAAAABs7V6yAAAAAI+KhRIAAAAAlTG/4wAAAABZ1hHPAAAAAK67NRIAAAAA0yllegAAAAAI4ggqAAAAAFd1r+QAAAAAVD5PxAAAAACEJwFEAAAAAKvEPaMAAAAA4Ac8IgAAAACU+wVZAAAAAP7+ERAAAAAAbIiBGQAAAADTnhCtAAAAAK2BQ7AAAAAARNHRCgAAAAAO9KD2AAAAAFw+dpgAAAAAUumYygAAAAAyZmdQAAAAAF2d4V0AAAAAi1G95AAAAADo8goqAAAAAMn9p/YAAAAA9DqeXwAAAADsX61fAAAAAMWtNyMAAAAA/vtYjQAAAACyRtmWAAAAALcTXfkAAAAATBowqwAAAAAPHzDoAAAAAAas7UQAAAAADp4FCwAAAADS2S6VAAAAAF0/vKIAAAAAydfHwQAAAAABPdmEAAAAAH+tcJoAAAAAole3tQAAAAByMpq0AAAAAIPW42YAAAAAnjM9OwAAAACr6xYxAAAAAM191jcAAAAAOFeNvQAAAAA6dAdHAAAAAKBuwRQAAAAAekXkOgAA

In [ ]:
def create_model(
    params: Dict[str, Any] = {}, default_model: ToyModel | None = None, *args, **kwargs
) -> ToyModel:
    density = params["Density"]
    relative_importance = params["Importance"]
    # random_seed = params["Random Seeds"]

    generator = Generator(device="mps").manual_seed(42)

    model = ToyModel(
        distribution=SparseUniform(
            N_FEATURES, p_active=density, device="mps", generator=generator
        ),
        ae=TiedLinearRelu(
            N_FEATURES,
            N_HIDDEN,
            generator=generator,
            device="mps",
        ),
        importances=relative_importance ** torch.arange(N_FEATURES),
        device="mps",
    )
    return model

In [ ]:
grid2 = ModelGrid(
    create_model,
    axes=[
        Axis(label="Density", values=densities),
        Axis(label="Importance", values=importances),
        # Axis(label="Random Seeds", values=random_seeds),
    ],
    cache_samples=True,
)

Building sample index: 100%|██████████| 10000/10000 [00:01<00:00, 7270.69model/s]


In [ ]:
grid2.fit(n_epochs=1_000)

Training:  41%|████      | 412/1000 [01:07<01:36,  6.10epoch/s]


KeyboardInterrupt: 

In [ ]:
def create_model_3(
    params: Dict[str, Any] = {}, default_model: ToyModel | None = None, *args, **kwargs
) -> ToyModel:
    density = params["Density"]
    relative_importance = 0.9 # params["Importance"]
    # random_seed = params["Random Seeds"]

    generator = Generator(device="mps").manual_seed(42)

    model = ToyModel(
        distribution=SparseUniform(
            N_FEATURES, p_active=density, device="mps", generator=generator
        ),
        ae=TiedLinearRelu(
            N_FEATURES,
            N_HIDDEN,
            generator=generator,
            device="mps",
        ),
        importances=relative_importance ** torch.arange(N_FEATURES),
        device="mps",
    )
    return model

In [ ]:
grid3 = ModelGrid(
    create_model_3,
    axes=[
        Axis(label="Density", values=densities),
    ],
    cache_samples=True,
)

Building sample index: 100%|██████████| 100/100 [00:00<00:00, 8095.08model/s]


In [ ]:
grid3.fit(n_epochs=1000)

Training: 100%|██████████| 1000/1000 [00:05<00:00, 168.41epoch/s]


In [ ]:
model_grid = ModelGrid(
    create_model,
    axes=[
        Axis(label="Density", values=densities),
        Axis(label="Importance", values=importances),
        # Axis(label="Random Seeds", values=random_seeds),
    ],
    cache_samples=False,
)

Validating Autoencoder: 100%|██████████| 10000/10000 [00:00<00:00, 541102.77model/s]


In [ ]:
model_grid.fit(n_epochs=1000)

Training: 100%|██████████| 1000/1000 [10:59<00:00,  1.52epoch/s]
